In [ ]:
import logfire

from medici.common.services.hybrid_search import HybridSearch
from medici.common.services.qdrant import QdrantStorageService
from medici.common.services.reranker import Reranker
from medici.common.utils.config import config
from medici.ingestion.chunking.chunker_factory import create_chunker
from medici.ingestion.chunking.chunking_config import ChunkingConfig
from medici.ingestion.embedding import EmbeddingService

In [ ]:
logfire.configure(service_name="qdrant_test")

In [ ]:
qt = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=config.VECTOR_SIZE,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [ ]:
em = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME, dimensions=config.EMBEDDING_DIMENSIONS
)

In [ ]:
content_list = "Python Machine Learning Interview Prep — Part 1\n\nPython, ML Fundamentals, Supervised/Unsupervised, Data Wrangling & Benchmarking\n\nTarget Role:\n\nPython ML Engineer |\n\nExperience:\n\n2–3 Years\n\n1."

In [ ]:
chunking_config = ChunkingConfig(type="recursive_character", size=512, overlap=64)
chunker = create_chunker(chunking_config)
chunks = chunker.chunk(content_list)

In [ ]:
em_chunk = await em.embed_chunks(chunks)

In [ ]:
await qt.ping()

In [ ]:
await qt.upsert_embedded_chunks(embedded_chunks=em_chunk)

In [ ]:
ques = [
    "Explain the difference between `deepcopy` and `copy` in the context of ML model parameters?"
]

In [ ]:
em_query = await em.embed_single(ques[0])

In [ ]:
q_search = await qt.search(query=ques[0], query_vector=em_query)

In [ ]:
q_search

In [ ]:
hydrid = HybridSearch(storage_service=qt, embedding_service=em)

In [ ]:
final = await hydrid.search(queries=ques)
print(len(final))
print(final[0])

In [ ]:
rerank = Reranker()

In [ ]:
top_result = await rerank.rerank(query=ques[0], candidates=final)

In [ ]:
top_result